In [ ]:
# Upgrade pip and install dependencies with DNS fix
!python -m pip install --upgrade pip
!echo "nameserver 8.8.8.8" > /etc/resolv.conf
!pip install --trusted-host pypi.org --trusted-host files.pythonhosted.org \
    --default-timeout=100 --retries=10 --no-cache-dir \
    flask==3.0.3 opencv-python==4.9.0.80 mediapipe==0.10.14 numpy==1.26.4 \
    pyngrok==7.1.6 python-dotenv==1.0.1
!ngrok authtoken your_ngrok_token

In [ ]:
# Create face_detection.py
%%writefile /kaggle/working/face_detection.py
import cv2
import mediapipe as mp
import numpy as np
import os
import logging
from flask import Flask, request, jsonify
from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

app = Flask(__name__)
load_dotenv()

mp_face_detection = mp.solutions.face_detection
face_detection = mp_face_detection.FaceDetection(min_detection_confidence=0.5)

images_path = "/kaggle/working/images"
os.makedirs(images_path, exist_ok=True)
known_faces = {}
for fname in os.listdir(images_path):
    if fname.endswith((".jpg", ".jpeg", ".png")):
        name = os.path.splitext(fname)[0]
        known_faces[name] = os.path.join(images_path, fname)

@app.route('/add_face', methods=['POST'])
def add_face():
    try:
        name = request.form['name']
        image = request.files['image']
        logger.info(f"Adding face: {name}")
        image_data = image.read()
        image_array = cv2.imdecode(np.frombuffer(image_data, np.uint8), cv2.IMREAD_COLOR)
        frame_rgb = cv2.cvtColor(image_array, cv2.COLOR_BGR2RGB)
        results = face_detection.process(frame_rgb)
        if results.detections:
            image_path = os.path.join(images_path, f"{name}.jpg")
            cv2.imwrite(image_path, image_array)
            known_faces[name] = image_path
            logger.info(f"Face saved: {image_path}")
            return jsonify({"success": True, "error": None})
        logger.warning("No face detected")
        return jsonify({"success": False, "error": "No face detected"}), 400
    except Exception as e:
        logger.error(f"Error adding face: {str(e)}")
        return jsonify({"success": False, "error": str(e)}), 500

@app.route('/detect_face', methods=['POST'])
def detect_face():
    try:
        logger.info("Detecting faces")
        image = request.files['image']
        image_data = image.read()
        frame = cv2.imdecode(np.frombuffer(image_data, np.uint8), cv2.IMREAD_COLOR)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_detection.process(frame_rgb)
        face_locations = []
        face_names = []
        if results.detections:
            for detection in results.detections:
                bbox = detection.location_data.relative_bounding_box
                h, w, _ = frame.shape
                x1, y1 = int(bbox.xmin * w), int(bbox.ymin * h)
                x2, y2 = int((bbox.xmin + bbox.width) * w), int((bbox.ymin + bbox.height) * h)
                face_locations.append((y1, x2, y2, x1))
                face_img = frame[y1:y2, x1:x2]
                name = "Unknown"
                for known_name, known_path in known_faces.items():
                    known_img = cv2.imread(known_path)
                    if known_img is not None and face_img.shape == known_img.shape:
                        diff = np.mean((face_img - known_img) ** 2)
                        if diff < 1000:
                            name = known_name
                            break
                face_names.append(name)
            logger.info(f"Detected {len(face_names)} faces")
        return jsonify({"locations": face_locations, "names": face_names})
    except Exception as e:
        logger.error(f"Error detecting face: {str(e)}")
        return jsonify({"success": False, "error": str(e)}), 500

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=8000)

In [ ]:
# Run Flask with ngrok
from pyngrok import ngrok
import os

public_url = ngrok.connect(8000).public_url
print(f"Face Detection API at: {public_url}")
os.system('python /kaggle/working/face_detection.py')